# Exercise 7 - Clustering

**Anggota Kelompok:**
1. M. Furqon Lutfi Septriandri (25031554192)
2. Akram Ali Fi Natarik (25031554212)

**Soal:** terapkan semua metode *clustering* pada teks hasil *feature engineering* dari exercise sebelumnya, lalu analisis hasilnya.

**Data (hasil pre-processing Exercise 4, sama seperti Exercise 5 dan 6):**
1. `detik_preprocessed.csv` - artikel detik.com (Bahasa Indonesia, 58 dokumen)
2. `jsonplaceholder_preprocessed.csv` - JSONPlaceholder `/posts` (teks dummy *lorem ipsum*, 100 dokumen)

**Fitur:** matriks **TF-IDF** dari Exercise 5

**Metode clustering:**
1. K-Means (elbow, silhouette score, word cloud)
2. Hierarchical Clustering (dendrogram, `AgglomerativeClustering` linkage *ward*, silhouette score, word cloud)
3. Perbandingan kedua metode dan kesimpulan

## 0. Install & Import Library

In [ ]:
%pip install matplotlib wordcloud

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn import metrics
from scipy.cluster.hierarchy import dendrogram, linkage

## 1. Load Data Hasil Pre-processing

In [ ]:
df_detik = pd.read_csv("detik_preprocessed.csv")
df_detik['prepro'] = df_detik['prepro'].fillna('')
df_detik.shape

In [ ]:
df_api = pd.read_csv("jsonplaceholder_preprocessed.csv")
df_api['prepro'] = df_api['prepro'].fillna('')
df_api.shape

## 2. Feature Engineering (TF-IDF dari Exercise 5)

In [ ]:
# Mengubah teks menjadi matriks TF-IDF
vectorizer_detik = TfidfVectorizer()
X_detik = vectorizer_detik.fit_transform(df_detik['prepro'])
df_FE_detik = pd.DataFrame(X_detik.toarray(), columns=vectorizer_detik.get_feature_names_out())

vectorizer_api = TfidfVectorizer()
X_api = vectorizer_api.fit_transform(df_api['prepro'])
df_FE_api = pd.DataFrame(X_api.toarray(), columns=vectorizer_api.get_feature_names_out())

df_FE_detik.shape, df_FE_api.shape

### Fungsi bantu
Dua fungsi kecil dipakai berulang: `top_terms` untuk melihat kata dengan rata-rata TF-IDF tertinggi di tiap cluster (sebagai ringkasan "isi" cluster), dan `tampilkan_wordcloud` untuk word cloud tiap cluster seperti di materi.

In [ ]:
def top_terms(df_FE, labels, n=5):
    rows = []
    for c in sorted(set(labels)):
        rata2 = df_FE[labels == c].mean().sort_values(ascending=False).head(n)
        rows.append({
            'Cluster': c,
            'Jumlah dokumen': int((labels == c).sum()),
            'Kata dominan': ", ".join(rata2.index)
        })
    return pd.DataFrame(rows)

In [ ]:
from wordcloud import WordCloud

def tampilkan_wordcloud(df, kolom_label, judul):
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    for c, ax in zip(sorted(df[kolom_label].unique()), axes.ravel()):
        long_string = ','.join(list(df[df[kolom_label] == c]['prepro'].values))
        wordcloud = WordCloud(background_color="white", max_words=5000, contour_width=3)
        wordcloud.generate(long_string)
        ax.imshow(wordcloud.to_image())
        ax.set_title(f'{judul} - cluster {c}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 3. Unsupervised: K-Means

### 3.1 Menentukan jumlah cluster (Elbow & Silhouette)

In [ ]:
k = range(1, 9)
inertia_detik = []
inertia_api = []

for i in k:
    km_model = KMeans(n_clusters=i, random_state=10)
    km_model.fit(df_FE_detik)
    inertia_detik.append(km_model.inertia_)

    km_model = KMeans(n_clusters=i, random_state=10)
    km_model.fit(df_FE_api)
    inertia_api.append(km_model.inertia_)

print(list(k))
print([round(x, 2) for x in inertia_detik])
print([round(x, 2) for x in inertia_api])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, inertia, judul in zip(axes, [inertia_detik, inertia_api], ['detik.com', 'JSONPlaceholder']):
    ax.plot(k, inertia, marker='o')
    ax.set_xlabel('k value')
    ax.set_ylabel('inertia')
    ax.set_title(f'Elbow - {judul}')
    ax.grid()
plt.tight_layout()
plt.show()

In [ ]:
hasil_k = []
for i in range(2, 9):
    lab_d = KMeans(n_clusters=i, random_state=10).fit(df_FE_detik).labels_
    lab_a = KMeans(n_clusters=i, random_state=10).fit(df_FE_api).labels_
    hasil_k.append({
        'k': i,
        'Silhouette detik': metrics.silhouette_score(df_FE_detik, lab_d, metric='euclidean'),
        'Silhouette jsonplaceholder': metrics.silhouette_score(df_FE_api, lab_a, metric='euclidean')
    })
pd.DataFrame(hasil_k).round(3)

Kurva elbow tidak memperlihatkan "siku" yang jelas dan silhouette score untuk semua nilai k sangat kecil (lihat analisis di bawah), sehingga tidak ada nilai k yang menonjol sebagai yang terbaik. Karena itu dipakai **k = 4** seperti contoh di materi, supaya hasilnya mudah dibandingkan antar metode dan antar dataset.

In [ ]:
n_clusters = 4

### 3.2 K-Means - dataset detik.com

In [ ]:
kM_detik = KMeans(n_clusters=n_clusters, random_state=10).fit(df_FE_detik)
kM_detik

In [ ]:
df_detik['Kmeans_label'] = kM_detik.labels_
df_detik[['prepro', 'Kmeans_label']].head()

**Silhouette Score**

In [ ]:
labels = kM_detik.labels_
metrics.silhouette_score(df_FE_detik, labels, metric='euclidean')

In [ ]:
top_terms(df_FE_detik, kM_detik.labels_)

**Word Cloud**

In [ ]:
tampilkan_wordcloud(df_detik, 'Kmeans_label', 'Kmeans detik')

### 3.3 K-Means - dataset JSONPlaceholder

In [ ]:
kM_api = KMeans(n_clusters=n_clusters, random_state=10).fit(df_FE_api)
df_api['Kmeans_label'] = kM_api.labels_
df_api[['prepro', 'Kmeans_label']].head()

**Silhouette Score**

In [ ]:
labels = kM_api.labels_
metrics.silhouette_score(df_FE_api, labels, metric='euclidean')

In [ ]:
top_terms(df_FE_api, kM_api.labels_)

**Word Cloud**

In [ ]:
tampilkan_wordcloud(df_api, 'Kmeans_label', 'Kmeans jsonplaceholder')

## 4. Unsupervised: Hierarchical Clustering

### 4.1 Dendrogram

In [ ]:
linkage_detik = linkage(df_FE_detik, method='ward', metric='euclidean')
plt.figure(figsize=(14, 5))
dendrogram(linkage_detik, no_labels=True)
plt.title('Dendrogram - detik.com')
plt.tight_layout()
plt.show()

In [ ]:
linkage_api = linkage(df_FE_api, method='ward', metric='euclidean')
plt.figure(figsize=(14, 5))
dendrogram(linkage_api, no_labels=True)
plt.title('Dendrogram - JSONPlaceholder')
plt.tight_layout()
plt.show()

In [ ]:
print("detik   - tinggi 5 penggabungan terakhir:", np.round(linkage_detik[-5:, 2], 2))
print("api     - tinggi 5 penggabungan terakhir:", np.round(linkage_api[-5:, 2], 2))

### 4.2 Agglomerative Clustering - dataset detik.com

In [ ]:
hierarchical_cluster = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
labels = hierarchical_cluster.fit(df_FE_detik)

pred_agc_detik = pd.Series(hierarchical_cluster.labels_)
df_detik['Ward_label'] = pred_agc_detik
df_detik[['prepro', 'Kmeans_label', 'Ward_label']].head()

**Silhouette Score**

In [ ]:
labels = pred_agc_detik
metrics.silhouette_score(df_FE_detik, labels, metric='euclidean')

In [ ]:
top_terms(df_FE_detik, pred_agc_detik.values)

**Word Cloud**

In [ ]:
tampilkan_wordcloud(df_detik, 'Ward_label', 'Hierarchical detik')

### 4.3 Agglomerative Clustering - dataset JSONPlaceholder

In [ ]:
hierarchical_cluster_api = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
labels = hierarchical_cluster_api.fit(df_FE_api)

pred_agc_api = pd.Series(hierarchical_cluster_api.labels_)
df_api['Ward_label'] = pred_agc_api
df_api[['prepro', 'Kmeans_label', 'Ward_label']].head()

**Silhouette Score**

In [ ]:
labels = pred_agc_api
metrics.silhouette_score(df_FE_api, labels, metric='euclidean')

In [ ]:
top_terms(df_FE_api, pred_agc_api.values)

**Word Cloud**

In [ ]:
tampilkan_wordcloud(df_api, 'Ward_label', 'Hierarchical jsonplaceholder')

## 5. Perbandingan K-Means vs Hierarchical

In [ ]:
perbandingan = pd.DataFrame({
    'Dataset': ['detik', 'detik', 'jsonplaceholder', 'jsonplaceholder'],
    'Metode': ['K-Means', 'Hierarchical (ward)', 'K-Means', 'Hierarchical (ward)'],
    'Silhouette': [
        metrics.silhouette_score(df_FE_detik, df_detik['Kmeans_label']),
        metrics.silhouette_score(df_FE_detik, df_detik['Ward_label']),
        metrics.silhouette_score(df_FE_api, df_api['Kmeans_label']),
        metrics.silhouette_score(df_FE_api, df_api['Ward_label'])],
    'Ukuran cluster': [
        sorted(df_detik['Kmeans_label'].value_counts().tolist(), reverse=True),
        sorted(df_detik['Ward_label'].value_counts().tolist(), reverse=True),
        sorted(df_api['Kmeans_label'].value_counts().tolist(), reverse=True),
        sorted(df_api['Ward_label'].value_counts().tolist(), reverse=True)],
})
perbandingan['Silhouette'] = perbandingan['Silhouette'].round(3)
perbandingan

**Kesepakatan antar metode** (Adjusted Rand Index: 1 = identik, sekitar 0 = sama seperti acak). Nomor cluster di K-Means dan Hierarchical hanya label, jadi dibandingkan lewat ARI, bukan lewat kesamaan nomornya.

In [ ]:
from sklearn.metrics import adjusted_rand_score

pd.DataFrame({
    'Dataset': ['detik', 'jsonplaceholder'],
    'ARI (K-Means vs Hierarchical)': [
        adjusted_rand_score(df_detik['Kmeans_label'], df_detik['Ward_label']),
        adjusted_rand_score(df_api['Kmeans_label'], df_api['Ward_label'])]
}).round(3)

In [ ]:
# silhouette untuk beberapa nilai k pada Hierarchical (sebagai pembanding tabel K-Means di bagian 3.1)
hasil_h = []
for i in range(2, 9):
    lab_d = AgglomerativeClustering(n_clusters=i, linkage='ward').fit(df_FE_detik).labels_
    lab_a = AgglomerativeClustering(n_clusters=i, linkage='ward').fit(df_FE_api).labels_
    hasil_h.append({
        'k': i,
        'Silhouette detik': metrics.silhouette_score(df_FE_detik, lab_d),
        'Silhouette jsonplaceholder': metrics.silhouette_score(df_FE_api, lab_a)
    })
pd.DataFrame(hasil_h).round(3)

## 6. Analisis & Kesimpulan

**Struktur cluster pada data ini lemah.**
- **Elbow:** inertia turun landai tanpa "siku". Pada detik, dari sekitar 55.9 (k=1) ke sekitar 45.3 (k=8); pada jsonplaceholder dari sekitar 83.6 ke sekitar 75.3, bahkan lebih datar.
- **Silhouette:** semua nilainya mendekati 0. K-Means k=4 mendapat sekitar 0.016 (detik) dan 0.002 (jsonplaceholder); Hierarchical k=4 sekitar 0.035 dan 0.010. Untuk semua k dari 2 sampai 8, nilai tertinggi hanya sekitar 0.055 (detik) dan 0.014 (jsonplaceholder). Nilai di sekitar 0 berarti dokumen hampir sama dekatnya dengan cluster sendiri dan cluster tetangganya, jadi pemisahan yang terbentuk tidak tegas.
- **Dendrogram:** tinggi penggabungan naik pelan-pelan (detik: sekitar 1.8 sampai 2.08 pada 5 penggabungan terakhir; jsonplaceholder: sekitar 1.69 sampai 1.84), tanpa lompatan yang menonjol yang bisa menandakan jumlah cluster alami.

**Penyebabnya ada di data, bukan di metodenya.**
- **detik:** dokumen sangat pendek (rata-rata sekitar 4 kata) sehingga sebagian besar dokumen tidak berbagi kata (sekitar 90% pasangan punya similarity 0 di Exercise 6). Di ruang TF-IDF yang hampir ortogonal, jarak antar dokumen cenderung hampir sama semua.
- **jsonplaceholder:** isinya *lorem ipsum* acak, dan kata dominan di semua cluster didominasi kata-kata filler yang sama (`et`, `aut`, `ut`, `qui`, `est` muncul berulang di berbagai cluster), jadi cluster-nya tidak bisa dibedakan secara tema. Tabel "kata dominan" menunjukkan hal ini.

**Perbandingan metode (k=4):**
- **K-Means** pada detik menghasilkan ukuran cluster 29 / 14 / 13 / 2: satu cluster tema jelas (`bahagia, bijak, syukur`), satu cluster kecil berisi 2 dokumen (`kemarin, fokus, misteri`), dan satu cluster besar berisi 29 dokumen campuran. 
- **Hierarchical (ward)** pada detik menghasilkan ukuran 37 / 10 / 6 / 5 dengan cluster kecil yang lebih tematik: `bahagia, syukur, senyum`; `laku, bentuk, keluh`; dan `lihat, dunia, ubah`. Sisanya (37 dokumen) menjadi cluster "sisa" yang tidak punya tema jelas. Silhouette-nya juga sedikit lebih tinggi daripada K-Means pada kedua dataset.
- **Kesepakatan antar metode rendah**: ARI sekitar 0.06 pada kedua dataset, artinya pengelompokan K-Means dan Hierarchical hampir tidak saling sepakat. Tema seperti "bahagia-syukur" memang muncul di kedua metode, tetapi anggota cluster dan ukurannya berbeda.
- Pada jsonplaceholder, kedua metode sama-sama tidak menghasilkan cluster yang bermakna (silhouette sekitar 0.002 dan 0.010).

**Kesimpulan:**
1. Clustering hanya menemukan pola kalau datanya memang punya struktur. Pada teks pendek (detik) hanya muncul beberapa kelompok tematik kecil di antara dokumen yang saling berbagi kata kunci; pada teks acak (jsonplaceholder) tidak ada struktur bermakna yang bisa ditemukan.
2. Hierarchical (ward) di sini sedikit lebih unggul dari K-Means dalam silhouette dan kejelasan tema cluster kecil, tetapi selisihnya kecil dan keduanya sama-sama berada di rentang nilai yang lemah.
3. Silhouette score yang rendah harus dibaca sebagai peringatan kualitas data/fitur, bukan sebagai bukti metodenya salah. Untuk hasil yang lebih baik bisa dicoba korpus yang lebih besar, dokumen yang lebih panjang, atau fitur embedding (Word2Vec/FastText) yang bisa menghubungkan dokumen tanpa kata yang sama.